In [7]:
import pandas as pd
import networkx as nx
import requests
import pubchempy as pcp
from xml.etree import ElementTree as ET
import time
import csv
from tqdm import tqdm
# ssh 242CD027@10.3.0.127

In [8]:
df_cg = pd.read_csv('edges/chemgene.csv')
df_cd = pd.read_csv('edges/chemdis.csv')
df_gd = pd.read_csv('edges/genedis.csv')

df_c = pd.read_csv('vocab/CTD_chemicals.csv')
df_d = pd.read_csv('vocab/CTD_diseases.csv')
df_g = pd.read_csv('vocab/CTD_genes.csv')

df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneSymbol'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseName'])  
df_gd = df_gd.drop_duplicates(subset=['GeneSymbol', 'DiseaseName'])

In [14]:
df_cg = df_cg[['GeneID','GeneSymbol']]

In [25]:
df_cg.head()
for s,d in zip(df_cg['GeneSymbol'],df_cg['GeneID']):
    if(type(s) != str):
        print(d)

45338


In [10]:
print(df_cg.info())
print(df_cd.info())
print(df_gd.info())



<class 'pandas.core.frame.DataFrame'>
Index: 812241 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        812241 non-null  object 
 1   ChemicalID          812241 non-null  object 
 2   CasRN               557625 non-null  object 
 3   GeneSymbol          812240 non-null  object 
 4   GeneID              812241 non-null  int64  
 5   GeneForms           809771 non-null  object 
 6   Organism            812241 non-null  object 
 7   OrganismID          812241 non-null  float64
 8   Interaction         812241 non-null  object 
 9   InteractionActions  812241 non-null  object 
 10  PubMedIDs           812241 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 74.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 3467034 entries, 0 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [ ]:
print(df_c.info())
print(df_d.info())
print(df_g.info())

In [13]:
print(len(df_cg['GeneID'].unique()))
print(len(df_cg['GeneSymbol'].unique()))
print(len(df_cg['ChemicalID'].unique()))
print(len(df_gd['GeneID'].unique()))
print(len(df_gd['GeneSymbol'].unique()))
print(len(df_gd['DiseaseID'].unique()))
print(len(df_cd['ChemicalID'].unique()))
print(len(df_cd['DiseaseID'].unique()))


28279
28279
10972
9112
9112
5860
17705
7284


In [3]:
G = nx.Graph()
chemicals = set(df_cg['ChemicalID'].unique()) | set(df_cd['ChemicalID'].unique())
genes = set(df_cg['GeneID'].unique()) | set(df_gd['GeneID'].unique())
diseases = set(df_cd['DiseaseID'].unique()) | set(df_gd['DiseaseID'].unique())


print(len(chemicals))
print(genes)
diseases = [dis[5:] for dis in diseases]
print(diseases)

17795
{np.int64(1), np.int64(2), np.int64(131076), np.int64(100130562), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(100859921), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(131096), np.int64(25), np.int64(26), np.int64(27), np.int64(24), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(729451), np.int64(43), np.int64(729453), np.int64(131118), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(58), np.int64(59), np.int64(60), np.int64(62), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(75), np.int64(131149), np.int64(81), np.int64(82), np.int64(100335701), np.int64(86), np.int64(87), np.int64(88), np.int64(89), np.int64(90),

In [ ]:
# df_c = df_c[['ChemicalID', 'ChemicalName', 'PubChemCID']].drop_duplicates(subset=['ChemicalID']).set_index('ChemicalID')
# df_d = df_d[['DiseaseID', 'DiseaseName', 'DiseaseSemanticType']].drop_duplicates(subset=['DiseaseID']).set_index('DiseaseID')
# df_g = df_g[['GeneID', 'GeneSymbol', 'GeneName']].drop_duplicates(subset=['GeneID']).set_index('GeneID')

def get_smiles_from_mesh_id(mesh_id,writer):
    """
    Fetch SMILES strings for a chemical given its MeSH ID.
    
    Parameters:
    mesh_id (str): MeSH ID, e.g., 'D001241' for Aspirin (without prefix if not included).
    
    Returns:
    list: List of canonical SMILES strings for linked compounds.
    """
    # Step 1: Link MeSH ID to PubChem CIDs using NCBI E-Utilities
    # Ensure MeSH prefix

    cids = []

    if(df_c['ChemicalID'].isin(["MESH:"+mesh_id]).any()):
        chemical_name = str(df_c[df_c['ChemicalID'] == "MESH:"+mesh_id]['ChemicalName']).split()[1]

    pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{chemical_name}/cids/TXT"
    pug_response = requests.get(pug_url)
    if pug_response.status_code == 200:
        cids = [cid.strip() for cid in pug_response.text.strip().split('\n') if cid.strip()]
        time.sleep(0.1)  # Rate limit (PubChem: 5 req/sec)
    else:
        # print(f"PubChem exchange failed: HTTP {pug_response.status_code}")
        return []
    
    if not cids:
        return []
    
    # Step 4: Fetch SMILES from CIDs
    smiles_list = []
    for cid in cids[:10]:  # Limit to first 10 to avoid overload
        try:
            compound = pcp.Compound.from_cid(int(cid))
            if compound and compound.connectivity_smiles:
                smiles_list.append(compound.connectivity_smiles)
        except Exception as e:
            print(f"Error fetching SMILES for CID {cid}: {e}")
        # time.sleep(0.25)  # Rate limit for pubchempy
    writer.writerow([mesh_id,smiles_list])
    return smiles_list

# Example usage

    # print(f"SMILES for MeSH {chem}: {smiles}")

with open('vocab/smiles.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['ChemicalID', 'SMILES'])
    smiles = {}
    for chem in tqdm(chemicals):
        get_smiles_from_mesh_id(chem,writer)

In [4]:
smil = pd.read_csv('vocab/smiles.csv')
smilchem = set(smil['ChemicalID'])
rem = chemicals - smilchem
print(len(rem))
print(rem)

4507
{'C481448', 'C000617791', 'C000613875', 'C110453', 'D015059', 'C082419', 'C017323', 'C006084', 'C562236', 'C583126', 'C451426', 'C073341', 'C031815', 'C069172', 'C569751', 'C529402', 'C000593259', 'C492470', 'C031571', 'C558581', 'C063509', 'C067259', 'D006843', 'C053479', 'C485711', 'C092006', 'C080194', 'C489827', 'C000612733', 'C435127', 'C023867', 'C000597136', 'C496953', 'C025153', 'C105338', 'D014238', 'C548688', 'C107821', 'C095911', 'C432684', 'C000601111', 'C087808', 'C008776', 'C517487', 'C080279', 'C009213', 'C044260', 'C006798', 'C030973', 'C000624218', 'C116689', 'C023366', 'C000706936', 'D013951', 'D028581', 'C070690', 'C100245', 'D000318', 'C572247', 'C572868', 'C107757', 'C410959', 'C577640', 'C562246', 'C045130', 'C118956', 'C035958', 'C011459', 'C001380', 'C007484', 'C031263', 'C577636', 'C042413', 'C092396', 'D016179', 'C526863', 'D006512', 'C000710058', 'C000614725', 'D004533', 'C047944', 'C071721', 'C000712072', 'C046635', 'D011761', 'C530698', 'C001687', 'C52

In [11]:
import csv
from tqdm import tqdm

def get_smiles_from_mesh_id(mesh_id):
    if(df_c['ChemicalID'].isin(["MESH:"+mesh_id]).any()):
        chemical_name = str(df_c[df_c['ChemicalID'] == "MESH:"+mesh_id]['ChemicalName']).split()[1]

    pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/name/{chemical_name}/sids/TXT"
    pug_response = requests.get(pug_url)
    if pug_response.status_code == 200:
        return [cid.strip() for cid in pug_response.text.strip().split('\n') if cid.strip()]
    else:
        return None


ids = ['C481448', 'C000617791', 'C000613875', 'C110453', 'D015059', 'C082419', 'C017323', 'C006084', 'C562236', 'C583126', 'C451426', 'C073341', 'C031815', 'C069172', 'C569751', 'C529402', 'C000593259', 'C492470', 'C031571', 'C558581', 'C063509', 'C067259', 'D006843', 'C053479', 'C485711', 'C092006', 'C080194', 'C489827', 'C000612733', 'C435127', 'C023867', 'C000597136', 'C496953', 'C025153', 'C105338', 'D014238', 'C548688', 'C107821', 'C095911', 'C432684', 'C000601111', 'C087808', 'C008776', 'C517487', 'C080279', 'C009213', 'C044260', 'C006798', 'C030973', 'C000624218', 'C116689', 'C023366', 'C000706936', 'D013951', 'D028581', 'C070690', 'C100245', 'D000318', 'C572247', 'C572868', 'C107757', 'C410959', 'C577640', 'C562246', 'C045130', 'C118956', 'C035958', 'C011459', 'C001380', 'C007484', 'C031263', 'C577636', 'C042413', 'C092396', 'D016179', 'C526863', 'D006512', 'C000710058', 'C000614725', 'D004533', 'C047944', 'C071721', 'C000712072', 'C046635', 'D011761', 'C530698', 'C001687', 'C523668', 'C005799', 'C570774', 'C561322', 'C051019', 'C555398', 'C561899', 'C502239', 'D009074', 'D000093842', 'C070840', 'C568680', 'C437043', 'D000952', 'C057916', 'D005740', 'C070295', 'D000090982', 'C090413', 'C517823', 'C000600690', 'C403966', 'C470768', 'D003544', 'C000433', 'C515439', 'D019818', 'C066209', 'C516694', 'C416650', 'C029748', 'C022895', 'C539975', 'C020922', 'C029383', 'C116824', 'C071194', 'C052901', 'C550436', 'C008629', 'C063007', 'C000605449', 'C000595926', 'C100167', 'C520945', 'C052026', 'C113831', 'C098580', 'C588130', 'C438312', 'C112292', 'C482402', 'D001865', 'C555019', 'C068117', 'C000594958', 'C558917', 'C433985', 'C468748', 'C569454', 'C508504', 'C528847', 'D020320', 'C519016', 'C494022', 'D009173', 'C019539', 'D043371', 'C568513', 'C477197', 'D005730', 'C047517', 'C007999', 'C013298', 'C466918', 'C020380', 'C032793', 'C024347', 'C583222', 'C502972', 'C109078', 'C000625506', 'C116441', 'C000590832', 'C017514', 'C582844', 'C568854', 'C488333', 'C105490', 'C488107', 'C546193', 'C500530', 'C052090', 'C526709', 'C585955', 'C542473', 'C080077', 'C512236', 'C513728', 'C497852', 'C517027', 'D006576', 'C507634', 'C507404', 'D062610', 'D017962', 'C042691', 'C089517', 'C504035', 'C058278', 'C544830', 'C419146', 'C470272', 'C052410', 'C016780', 'C111309', 'C547594', 'C103474', 'C000710673', 'C550724', 'C508075', 'C117789', 'C570894', 'C075656', 'C573233', 'C000593032', 'C045037', 'D002614', 'C000722764', 'C013102', 'C055494', 'C048436', 'C000596770', 'D062385', 'D019268', 'C047727', 'D017942', 'C082739', 'D005999', 'D015288', 'D001383', 'C107820', 'C476441', 'C047825', 'C404207', 'C022999', 'D000621', 'C042431', 'C075773', 'C004999', 'C480398', 'C550047', 'C008374', 'C579558', 'D002635', 'C110793', 'C000606289', 'D000960', 'C520612', 'C481758', 'C031161', 'C520846', 'C070383', 'C544380', 'C096325', 'C000630584', 'C025964', 'C093700', 'C006796', 'C070070', 'C016519', 'C435583', 'C586072', 'C578599', 'C000633464', 'C038717', 'C064183', 'D001205', 'C533622', 'C035377', 'D055768', 'C086349', 'D034442', 'C510259', 'C504151', 'C569670', 'C117776', 'C026978', 'D047072', 'D004744', 'D017500', 'C011369', 'C029435', 'C118949', 'C110057', 'C570532', 'C000595230', 'D020097', 'D018733', 'C573097', 'D063308', 'C574037', 'C471867', 'C000629984', 'C074636', 'C548544', 'D002109', 'C110196', 'C480958', 'D008764', 'D036563', 'C113642', 'D011142', 'C481663', 'C500368', 'C041348', 'C000589030', 'C067410', 'C569674', 'C000708084', 'C074107', 'C027387', 'C025483', 'C456402', 'C101044', 'C000626357', 'C552103', 'C121726', 'C023036', 'C043051', 'D011075', 'C040996', 'C015668', 'C544865', 'C032761', 'C052853', 'C000612000', 'C430230', 'C035419', 'C544151', 'C552435', 'C025485', 'C506392', 'C000614724', 'C035353', 'C000632506', 'C097798', 'C576268', 'C492995', 'C000604517', 'C451162', 'C058064', 'C523415', 'C033851', 'C121756', 'C572625', 'C009000', 'C572065', 'D004743', 'D015074', 'D011687', 'C561592', 'D000701', 'C576262', 'C028000', 'C487679', 'C534147', 'C464660', 'C000604698', 'C118308', 'C561240', 'C019116', 'C071693', 'C560357', 'C575803', 'C058787', 'C039685', 'D019900', 'C051208', 'D006165', 'C032688', 'D003064', 'D019158', 'C474253', 'C571782', 'C058216', 'C068572', 'C030412', 'C109860', 'C012842', 'C533807', 'C533150', 'C000604463', 'C010285', 'D006897', 'D058955', 'C521311', 'C559869', 'C039964', 'C001918', 'C045762', 'C575894', 'C083377', 'D010833', 'C104543', 'C060988', 'C524699', 'C529372', 'C034341', 'D053245', 'C108892', 'C505382', 'C066926', 'C546412', 'C117663', 'C528564', 'C466797', 'C476033', 'C475637', 'C581040', 'C529296', 'C442659', 'C495872', 'C087926', 'C524716', 'C081310', 'C547088', 'C500067', 'D003999', 'C095446', 'C516685', 'C060229', 'C092926', 'C000592751', 'C053876', 'C568517', 'C080871', 'C478058', 'C525538', 'C420606', 'C000482', 'C499782', 'C584426', 'C522461', 'C546141', 'C459539', 'C560271', 'C093344', 'C000604852', 'C494665', 'C570695', 'C036150', 'D002793', 'C572649', 'C101670', 'C561234', 'C058539', 'C421124', 'C000720111', 'C110743', 'C046983', 'C489138', 'C000593592', 'C540688', 'C466420', 'C545121', 'C000728228', 'C530954', 'C539020', 'C035381', 'C532254', 'C000228', 'C001669', 'D020148', 'C549043', 'C514340', 'C502354', 'C012729', 'C051595', 'C030521', 'D019483', 'C494607', 'D020126', 'C013721', 'D014765', 'C558012', 'D009609', 'D026361', 'C029900', 'D018501', 'C071262', 'C060368', 'C583459', 'D008798', 'C542041', 'C530610', 'C000592767', 'C030583', 'C062198', 'C426703', 'C554954', 'C573861', 'C059975', 'C584816', 'C528663', 'C011363', 'D016308', 'C492322', 'C121203', 'C558746', 'C028108', 'C443808', 'D000074263', 'D005648', 'C487689', 'C507635', 'C546624', 'C040308', 'C478738', 'C549541', 'C008180', 'C007789', 'C002963', 'C066624', 'C047285', 'D016655', 'C570583', 'D000889', 'C412811', 'C073946', 'C552240', 'C571443', 'C000629279', 'C458508', 'C111679', 'D010191', 'C562225', 'C533760', 'C568948', 'C030290', 'D017878', 'C573544', 'D064586', 'C570295', 'C010247', 'C548902', 'C535212', 'C557784', 'C548271', 'C091682', 'C568605', 'C083781', 'D000897', 'C515299', 'C112040', 'C000619336', 'D005672', 'D014560', 'C410128', 'C047832', 'C559491', 'C570688', 'D004546', 'C101574', 'C118852', 'D008095', 'C572286', 'C508178', 'C067523', 'C076256', 'C000612922', 'C585486', 'C088489', 'C012482', 'C477055', 'C026087', 'D064209', 'C492719', 'D000040', 'C000620620', 'C053753', 'C436447', 'D000077442', 'C568516', 'C022022', 'C547957', 'C054919', 'C035884', 'C000721092', 'C012986', 'D013549', 'C073838', 'C030371', 'C076809', 'C407311', 'C513930', 'C517980', 'C543889', 'C547455', 'C524550', 'C067982', 'D000069479', 'C031599', 'C071788', 'C534004', 'C064545', 'C085977', 'C561233', 'C529548', 'C037112', 'C000623387', 'C068457', 'C070595', 'C534240', 'C040405', 'C487113', 'D012905', 'C027492', 'D019162', 'C572555', 'C542472', 'C488676', 'C497468', 'C562348', 'D000068196', 'C065419', 'C000624602', 'C501339', 'D001897', 'C018021', 'D037341', 'C502768', 'D005460', 'C037913', 'C062894', 'C018250', 'C000713447', 'C442100', 'C011724', 'C482084', 'C040735', 'C000611388', 'C548487', 'C448659', 'C547321', 'C085590', 'C522391', 'C569176', 'C432168', 'C000628396', 'C074544', 'C116322', 'C000712174', 'D000338', 'C028498', 'C024871', 'C082486', 'C500727', 'C581540', 'C487834', 'C506606', 'C549376', 'C553231', 'C044301', 'C483824', 'C469905', 'C500237', 'C042630', 'D002257', 'C114969', 'C046393', 'C007180', 'C552704', 'D009227', 'C479069', 'C120158', 'D000069463', 'C064515', 'C511855', 'C505532', 'C506643', 'C497687', 'C054715', 'D017636', 'C471268', 'C019419', 'D004043', 'C100395', 'C021597', 'C038649', 'C467540', 'C587938', 'C501048', 'D052157', 'C048008', 'D001095', 'C043332', 'C003964', 'C088674', 'C106973', 'D012910', 'C586321', 'C085754', 'C553587', 'C432745', 'C552770', 'C084485', 'C065413', 'C507255', 'C009684', 'C475427', 'C539157', 'C552934', 'C515616', 'D005345', 'C072121', 'C500617', 'C013082', 'C083522', 'C507351', 'C031517', 'C063124', 'C000481', 'C416653', 'C509799', 'D000697', 'D005565', 'D003840', 'C000617180', 'C000720073', 'C456547', 'C014168', 'C000611296', 'C545833', 'C000620474', 'C494380', 'C502306', 'C065380', 'C502471', 'C087775', 'C000601816', 'C113661', 'C065046', 'C012382', 'D020100', 'C063968', 'D015772', 'D054804', 'C001205', 'C026562', 'D014013', 'C473711', 'C051263', 'C062025', 'C088450', 'C070375', 'C000593130', 'D004123', 'C097949', 'C569292', 'C051597', 'D009822', 'C064218', 'C008122', 'C028716', 'C111258', 'C008400', 'C464947', 'D003277', 'D014148', 'D000097245', 'C499366', 'C588402', 'C000622041', 'C040155', 'C020481', 'C079056', 'C002373', 'C038780', 'C570445', 'C068250', 'C023699', 'C004632', 'C096329', 'C033158', 'C069711', 'C067584', 'C053205', 'D001905', 'C107145', 'C527637', 'C101202', 'C012776', 'D009544', 'C484309', 'C499768', 'C439392', 'C457232', 'C000706872', 'C562244', 'C561055', 'D000942', 'C539199', 'C571662', 'C056510', 'D019422', 'C011465', 'C087806', 'C000730419', 'C585652', 'C000632388', 'C561226', 'C010925', 'C034182', 'C000611248', 'C503100', 'C100143', 'C538773', 'C583716', 'C534880', 'C521094', 'C070492', 'C024873', 'C515005', 'C046285', 'C570789', 'C000719634', 'C483839', 'D003652', 'D020916', 'C480819', 'C080955', 'C119006', 'C004867', 'C482289', 'C000718871', 'C029658', 'C001736', 'C509618', 'C570862', 'C096324', 'C575009', 'C562239', 'C013598', 'C000598200', 'D006401', 'C568519', 'D019216', 'C509502', 'C024915', 'D013205', 'C035891', 'C471581', 'C501946', 'C035953', 'C012961', 'C560081', 'C569671', 'D015123', 'C009265', 'C113145', 'C033094', 'C073220', 'D014574', 'C000626766', 'C524733', 'C558663', 'C064093', 'C031660', 'C085516', 'C514869', 'C522525', 'C009504', 'C000657144', 'C529144', 'D005659', 'C054649', 'C515166', 'D005707', 'D006219', 'C100416', 'C000620621', 'D020875', 'C056574', 'C009524', 'D008042', 'C446602', 'C548427', 'C095607', 'C476378', 'C505166', 'C076950', 'C044630', 'C057741', 'C400184', 'C015392', 'C405699', 'C549133', 'C031345', 'C574057', 'C460579', 'D012739', 'C532038', 'C105509', 'D000068838', 'C000591165', 'C054893', 'C521339', 'C114289', 'C532757', 'C060533', 'D012116', 'C087705', 'C030691', 'D013024', 'C574017', 'D012531', 'D005680', 'C530049', 'C406867', 'C053316', 'C498554', 'C576869', 'C004809', 'C475739', 'C053476', 'C112356', 'C000720114', 'C451779', 'C562255', 'C000596772', 'C576645', 'C480486', 'C011670', 'C522219', 'C505765', 'C520230', 'C000612923', 'C401312', 'D010072', 'C077642', 'C500321', 'C560356', 'C558047', 'C095007', 'C507129', 'C511222', 'C570662', 'D006571', 'C000620557', 'C078131', 'C112355', 'C069295', 'C522118', 'C541120', 'C412012', 'C486184', 'C407042', 'C105686', 'C545423', 'D007287', 'C496751', 'C524061', 'D001725', 'C412469', 'C119130', 'C453902', 'C005954', 'D007252', 'C417190', 'C403088', 'C526957', 'C569669', 'C045607', 'C544516', 'C055172', 'D008301', 'C043103', 'C045958', 'C040776', 'C069537', 'C555225', 'C518667', 'C000597426', 'C009674', 'C510090', 'C028800', 'C053177', 'C514579', 'C525872', 'C560297', 'C105293', 'C408128', 'C070417', 'C034172', 'C109643', 'C000598201', 'C045427', 'C568289', 'C097599', 'C562068', 'C090019', 'C496347', 'C015320', 'C054683', 'D003276', 'C071541', 'C027172', 'C025136', 'C015407', 'C551631', 'D006846', 'C419924', 'C501634', 'C005460', 'C000622214', 'C078284', 'C533671', 'C025637', 'C044922', 'D008063', 'C042080', 'C403753', 'D007166', 'C521776', 'C533004', 'C080974', 'C004860', 'C540456', 'C075569', 'C434605', 'C558524', 'D003373', 'C446919', 'D011838', 'C000604795', 'C119063', 'C040168', 'C555408', 'C400142', 'C559943', 'C452642', 'C031469', 'C000597780', 'D014502', 'C439329', 'C000604281', 'C049807', 'C530804', 'C078255', 'C046386', 'C010352', 'D000936', 'C105726', 'C503102', 'C478479', 'C530768', 'C006358', 'D003993', 'C025087', 'C038300', 'C109496', 'C054551', 'C532668', 'C521104', 'D016226', 'D013107', 'C117813', 'C092756', 'C026163', 'D006845', 'C081972', 'C000625749', 'C032948', 'D019274', 'C001363', 'C015357', 'C476103', 'C000588914', 'C072593', 'C065087', 'D009943', 'D005034', 'C529061', 'C512486', 'C000720117', 'C487933', 'C022834', 'C015144', 'C015138', 'C086401', 'C505200', 'C520647', 'C400080', 'C470794', 'D011099', 'C525762', 'C031442', 'C408387', 'C074681', 'C540740', 'C570584', 'C000622812', 'C477506', 'C002630', 'D014107', 'C554850', 'C098408', 'C023710', 'D044947', 'C424927', 'C048763', 'C047116', 'D000074262', 'C480541', 'C539236', 'C022616', 'C100261', 'D005456', 'C504746', 'C067513', 'C068408', 'C587256', 'C000591895', 'C506540', 'C018392', 'D009944', 'C517943', 'C040879', 'C547926', 'C033502', 'C556168', 'C540576', 'C548780', 'C512456', 'C546793', 'C099958', 'C553471', 'C542801', 'D000080506', 'C439205', 'C014596', 'C553676', 'C001073', 'D014108', 'C053063', 'C087490', 'C031563', 'C559853', 'C554269', 'C405943', 'C539676', 'C561635', 'C449119', 'C542815', 'C553486', 'C000632706', 'C408069', 'C027876', 'D036702', 'C533485', 'C520147', 'C030298', 'C540167', 'C552735', 'D003224', 'C096344', 'C513115', 'C000720488', 'D019319', 'C008859', 'D000978', 'C101948', 'D002726', 'C472668', 'D010169', 'C094710', 'D000069499', 'C546571', 'C047651', 'D015097', 'C492721', 'C059042', 'C000621655', 'C104872', 'C522027', 'D007480', 'C036585', 'D005617', 'D017382', 'C549569', 'C552420', 'C518786', 'C039783', 'C492572', 'C087181', 'C008363', 'D058915', 'C086492', 'C000612732', 'D000929', 'C403132', 'C020889', 'C570402', 'C484848', 'C498328', 'C479122', 'C099564', 'C083374', 'C008216', 'C037974', 'C402107', 'C543853', 'C454039', 'C516779', 'C013123', 'D007736', 'C575715', 'C466792', 'C478160', 'C001318', 'C534532', 'C063159', 'C487484', 'C513228', 'C555658', 'C572961', 'D010463', 'C013785', 'C048581', 'C007842', 'C479799', 'D002093', 'C568783', 'C039642', 'D010129', 'C104288', 'C510261', 'C078928', 'C503772', 'C530475', 'C545239', 'C569478', 'D051219', 'C468270', 'C550903', 'C063942', 'C415424', 'C526437', 'C003218', 'C000726650', 'C583913', 'C001220', 'C090034', 'C514718', 'C036835', 'C531537', 'C009091', 'D020025', 'C542083', 'C083136', 'D010712', 'C546413', 'C059444', 'D049990', 'C539427', 'C482404', 'C000627785', 'D011084', 'C000621942', 'D016166', 'C000628281', 'C051869', 'C011008', 'C029091', 'C532601', 'C099630', 'C489023', 'C569194', 'C010389', 'C057006', 'C549913', 'C024137', 'C551172', 'D006842', 'C555640', 'C459379', 'C051760', 'C009622', 'C423842', 'C066776', 'C552046', 'C546905', 'C524135', 'C120076', 'C569763', 'C064755', 'C097008', 'C069293', 'C075222', 'C081914', 'C043060', 'D018809', 'C474958', 'C000590771', 'C089549', 'C072790', 'C058818', 'C000613347', 'C535076', 'C046170', 'C540165', 'C086089', 'C572052', 'C017302', 'C010071', 'D014862', 'C032808', 'D006812', 'C058895', 'C528044', 'C000592353', 'C500584', 'C060203', 'D008076', 'D013259', 'D053918', 'C000591546', 'C000616972', 'D010456', 'C000613560', 'D006858', 'C077313', 'C110258', 'C480543', 'C096501', 'D016718', 'C014258', 'D053773', 'C461384', 'C488795', 'C091766', 'C047210', 'C521312', 'C580814', 'C012019', 'C521167', 'C559381', 'C513867', 'D012968', 'D007656', 'C545122', 'C000589285', 'C073661', 'D058607', 'C017095', 'D017498', 'C007500', 'C562251', 'D002301', 'C007829', 'C030826', 'D019817', 'C005210', 'C550356', 'C041914', 'C505455', 'C493898', 'C484864', 'D015306', 'C084904', 'D010853', 'C585813', 'C437028', 'C054121', 'D003279', 'C036808', 'C068582', 'C030272', 'C008324', 'C559860', 'C031463', 'C014202', 'C021012', 'C517284', 'C441720', 'C459559', 'C017725', 'C033125', 'D014527', 'C439425', 'C447943', 'C026226', 'C060296', 'C518399', 'C509969', 'C429269', 'D007302', 'C096910', 'C033705', 'C584427', 'C477931', 'C525846', 'C046418', 'C568964', 'C094477', 'C029764', 'C521406', 'C499326', 'C000626350', 'C016534', 'C013499', 'C079148', 'C000596409', 'C495871', 'C433771', 'C517552', 'C573096', 'C021091', 'C474019', 'C512044', 'C000626787', 'C016050', 'C034832', 'D006256', 'C584749', 'D056831', 'C521767', 'C525191', 'D058785', 'C008911', 'C551528', 'D013735', 'C103119', 'C550328', 'D015085', 'C416849', 'C473217', 'D001888', 'D008627', 'C010715', 'C517041', 'C561057', 'D000073417', 'C026424', 'C039222', 'C000609387', 'C071924', 'C577756', 'C000604237', 'C118756', 'C000626967', 'C493238', 'C475726', 'C531230', 'C433825', 'D042341', 'C480140', 'C011107', 'C014871', 'C095429', 'D015124', 'C516400', 'D003683', 'C041398', 'D007513', 'C494667', 'D019380', 'C034527', 'C558635', 'C034554', 'C526145', 'C017321', 'D019814', 'C472511', 'C061242', 'C466859', 'D000806', 'D004791', 'D000317', 'C030530', 'D017127', 'C570531', 'C023013', 'C083486', 'C121292', 'C483032', 'C420084', 'C006854', 'C094643', 'C028959', 'C553295', 'C119626', 'C033266', 'C520255', 'C568442', 'C034332', 'C504100', 'C581581', 'C090725', 'D052203', 'C099813', 'C012637', 'C047971', 'C026346', 'C071595', 'C047011', 'C512968', 'D015126', 'D052244', 'C472086', 'C524256', 'C040750', 'C071005', 'C582717', 'C519880', 'C012639', 'D018639', 'C572189', 'C083610', 'C008039', 'C079055', 'C512438', 'C408362', 'C100716', 'C544923', 'C521292', 'D044946', 'C000594496', 'C553039', 'C540615', 'C000611292', 'D004228', 'C117711', 'C050847', 'C077194', 'C020946', 'C072610', 'C496216', 'C568020', 'C499970', 'C110820', 'C014682', 'C588229', 'C425227', 'C007590', 'C476829', 'C120401', 'C562247', 'D058612', 'C571099', 'C000595015', 'C119066', 'C526719', 'C004503', 'C548901', 'D018350', 'C050127', 'C497150', 'C006821', 'C402921', 'C000598422', 'C095803', 'C009355', 'C034096', 'C000611872', 'C081654', 'C110772', 'C574204', 'C042140', 'C556862', 'C524315', 'C071644', 'D015815', 'C000624612', 'C446520', 'C017032', 'D006573', 'C527851', 'C009497', 'C552888', 'C000589147', 'C432910', 'C520233', 'C106053', 'C558744', 'C470795', 'C525146', 'C038193', 'D019496', 'C584306', 'C034039', 'D014642', 'C530888', 'C000602261', 'D008455', 'C487936', 'C000989', 'C558638', 'C473625', 'C561917', 'C495366', 'C115950', 'C095932', 'C548783', 'C000730511', 'C000612304', 'C026359', 'C014815', 'C034584', 'C511951', 'C476326', 'C000612164', 'C407502', 'C477819', 'C000619244', 'C000599178', 'C561619', 'D005227', 'C095756', 'C412383', 'C419158', 'C442201', 'C090203', 'C477694', 'C560298', 'D019304', 'C094849', 'C582837', 'C561593', 'C111570', 'C519817', 'C102911', 'C539751', 'C534193', 'D057846', 'C000718387', 'C032005', 'C546706', 'C011664', 'C078986', 'C120261', 'D042462', 'C531525', 'C012046', 'D047348', 'C063087', 'C115284', 'C047721', 'C520026', 'C570407', 'C077579', 'C023599', 'D014970', 'C471009', 'C068409', 'C551468', 'D010205', 'C570543', 'C473640', 'D066248', 'D012663', 'C000611642', 'C005959', 'C521270', 'D013260', 'C577944', 'D045930', 'C521108', 'C087630', 'C486469', 'D004167', 'C579534', 'C055987', 'C502234', 'C047521', 'C029620', 'C076178', 'C014163', 'C561058', 'C439285', 'C518471', 'C499781', 'C495657', 'C484999', 'C108959', 'C016276', 'C035057', 'C445259', 'C570539', 'C103788', 'C416837', 'C548173', 'C119207', 'C084198', 'C101879', 'C579032', 'C000718376', 'C558727', 'C022893', 'C044959', 'C084981', 'C000628228', 'D011735', 'C032208', 'D010667', 'C079393', 'D048729', 'C412308', 'C116348', 'C522862', 'D013388', 'C571720', 'D019377', 'C045849', 'C009862', 'C008069', 'C583365', 'C088826', 'C458582', 'C489776', 'C095324', 'C531164', 'C521487', 'C023023', 'C082032', 'C500323', 'C561157', 'C511282', 'C079426', 'D000080545', 'C504089', 'C079315', 'C009900', 'C120576', 'C027762', 'C042720', 'C097963', 'C070967', 'C030514', 'C519396', 'C496641', 'C569107', 'D010893', 'D014285', 'C402179', 'D005343', 'C519184', 'C006863', 'C581034', 'C442830', 'C406527', 'C010642', 'D000894', 'C552033', 'C029795', 'C523615', 'D001961', 'C107873', 'D009963', 'C507164', 'C559609', 'D006495', 'C029729', 'C428264', 'D002791', 'C502015', 'C449037', 'C421642', 'C031183', 'C532268', 'D000077611', 'C020175', 'D012795', 'C479228', 'C585643', 'C034944', 'C473200', 'C516573', 'C540967', 'C545376', 'C103851', 'C080121', 'C551961', 'C031149', 'C022990', 'D014151', 'D052179', 'C511224', 'C416782', 'C524539', 'C000608699', 'C474829', 'C031071', 'C420062', 'D013655', 'C439952', 'C574336', 'C121954', 'D000069480', 'C502530', 'C000591713', 'C515661', 'C518727', 'D016682', 'C432503', 'C000605226', 'C030669', 'C057354', 'D017325', 'C510788', 'C546012', 'D058671', 'C002656', 'C499730', 'C556220', 'C093027', 'C024709', 'C121734', 'C523471', 'C585271', 'C518913', 'C051140', 'C042349', 'C097003', 'C406734', 'C102487', 'C088464', 'C097962', 'C024535', 'C515676', 'C552826', 'C011395', 'C111238', 'C107086', 'C067648', 'C456029', 'C559866', 'C545140', 'C533599', 'C558288', 'C507889', 'C118156', 'C489151', 'C518710', 'C475571', 'C033729', 'C000721052', 'C000623021', 'C492908', 'C064578', 'D058666', 'C500627', 'C014612', 'C477861', 'C045392', 'C050086', 'C055161', 'C048074', 'C061282', 'C105319', 'C522246', 'C509089', 'C422416', 'C547186', 'C028290', 'C554181', 'C118527', 'C499293', 'C087257', 'C479555', 'C550088', 'D006511', 'C452424', 'C085074', 'C508870', 'C046941', 'C000611379', 'C451219', 'C504153', 'C050592', 'C517828', 'C008919', 'C550670', 'C541824', 'C040995', 'C101618', 'C000995', 'C424802', 'C069724', 'C548433', 'C024033', 'C569673', 'C054250', 'C044565', 'C561054', 'D061945', 'D014118', 'C111046', 'C470178', 'D000893', 'C494243', 'C497385', 'C048972', 'C519463', 'C442842', 'D004603', 'C482267', 'D056810', 'C066716', 'D000981', 'C585831', 'C539241', 'C030990', 'C030610', 'C549041', 'D014677', 'C063456', 'C521392', 'D011847', 'C541506', 'C000720115', 'C480354', 'D017320', 'D034341', 'C483690', 'D015117', 'C000604982', 'C074183', 'D010567', 'D019301', 'D000319', 'C094059', 'D001125', 'C568912', 'C107238', 'D009930', 'C583223', 'C546561', 'C489957', 'D011310', 'C570200', 'C027579', 'C558048', 'D003287', 'D003436', 'C519714', 'C527855', 'C037657', 'C524250', 'C459179', 'C108123', 'C482801', 'C017133', 'C023020', 'C528740', 'C005101', 'C403253', 'C000621168', 'C057270', 'C000601996', 'C067795', 'D008528', 'C115774', 'C441070', 'C049508', 'C017856', 'C009135', 'C522903', 'C004822', 'C034153', 'C549498', 'C558275', 'C446920', 'C468797', 'C096274', 'C048213', 'C014117', 'C089799', 'C415363', 'C050715', 'C059835', 'C499807', 'C071275', 'C027561', 'C586710', 'C505055', 'C062567', 'D052578', 'C019370', 'C019468', 'C037740', 'C034219', 'D006603', 'C044855', 'D039741', 'D002084', 'C029892', 'C447621', 'C520041', 'C510898', 'C523027', 'C042533', 'C091518', 'C045858', 'C027746', 'D000069459', 'C017970', 'C510843', 'C549543', 'C530734', 'C523097', 'C039537', 'C544393', 'C000625668', 'C502971', 'C034171', 'C104302', 'C572491', 'D001371', 'C569748', 'C478626', 'C571169', 'C436284', 'D005228', 'D001496', 'C024794', 'D003701', 'C572783', 'D028321', 'C010229', 'C030242', 'C558529', 'C099950', 'C000654633', 'C555857', 'C518849', 'C066049', 'C544847', 'C011978', 'C062458', 'C067722', 'C571873', 'D015102', 'C040048', 'C438686', 'C095591', 'C000716953', 'C120227', 'C117710', 'C062889', 'C433814', 'C000721090', 'C584311', 'C568535', 'C046833', 'C008228', 'C081331', 'C500959', 'C043215', 'C416927', 'C534222', 'C000622612', 'D018796', 'C423264', 'D017638', 'C051743', 'C079415', 'C096092', 'C000597819', 'C511303', 'C523983', 'C010471', 'C002999', 'C000589286', 'C526583', 'C508470', 'C507420', 'C121396', 'C117416', 'C008047', 'C000613964', 'C553914', 'C499827', 'C118950', 'C442865', 'C549106', 'C585906', 'C576297', 'C000620716', 'C543846', 'C020300', 'C009198', 'C545830', 'C524478', 'C063918', 'C549189', 'C040488', 'C485107', 'C438612', 'C587734', 'C004821', 'D014641', 'C458523', 'C523965', 'C526550', 'C107010', 'C000607343', 'C561372', 'D000077182', 'C000655644', 'C088115', 'D009942', 'D013744', 'C041942', 'C544671', 'C500007', 'C066185', 'D012856', 'C480030', 'C553473', 'D010938', 'C081174', 'D004040', 'C477185', 'C010590', 'D013473', 'C073279', 'C053972', 'C118301', 'C020979', 'C474071', 'C077059', 'C097004', 'C000599672', 'C507046', 'C051594', 'C044846', 'C070620', 'C036658', 'D052638', 'C063822', 'C000730654', 'C561231', 'C551617', 'C553674', 'C435044', 'C029092', 'C452229', 'C006486', 'C121978', 'C412895', 'C043081', 'D053497', 'C098434', 'C418233', 'C528423', 'C109986', 'D000073943', 'C022959', 'C073734', 'C411885', 'C000608473', 'C549240', 'C550144', 'C056934', 'C558050', 'D016173', 'D013184', 'C492909', 'C076944', 'C568376', 'C000716307', 'C041711', 'D011663', 'C064788', 'C108904', 'C000720250', 'C023600', 'C579035', 'C000710227', 'C500238', 'C569690', 'D022542', 'C576036', 'C086854', 'C000601815', 'D017319', 'C511991', 'C000624588', 'C492672', 'C071739', 'C494124', 'C025421', 'C040403', 'C492847', 'D000075182', 'C000599179', 'C585745', 'C547156', 'D004051', 'C527688', 'C557705', 'C031274', 'C571256', 'C120963', 'C488288', 'C008366', 'C472634', 'C000628791', 'C082182', 'C514145', 'C000588561', 'C573282', 'C000713259', 'C534345', 'C402306', 'D000090983', 'C492919', 'C521787', 'C409874', 'C570530', 'C512045', 'C496950', 'C454400', 'C118309', 'C558243', 'C530802', 'C553472', 'C475470', 'D000080044', 'C104519', 'C572775', 'D009569', 'D013871', 'C116619', 'C054996', 'D005230', 'C509276', 'C018264', 'C000599362', 'C033772', 'C493218', 'C502982', 'C555407', 'C501280', 'C000516', 'C520164', 'C544309', 'C000588958', 'D007155', 'D010909', 'C524875', 'C025913', 'C570037', 'C092038', 'C575690', 'C058176', 'C465664', 'C418824', 'C418762', 'C000613963', 'D012265', 'D011131', 'C434926', 'C003058', 'C109992', 'C090068', 'C500716', 'C101851', 'C505150', 'D011089', 'C539240', 'C532657', 'C047175', 'C000629597', 'C010210', 'C104459', 'C558286', 'D014317', 'C535211', 'C037881', 'C561228', 'C581004', 'C075750', 'C410009', 'C024023', 'D000070416', 'C011582', 'C411671', 'C521279', 'C411134', 'C408042', 'C570208', 'C089813', 'C586085', 'C025004', 'C477182', 'D022621', 'C425644', 'C414353', 'C509927', 'C561931', 'C511179', 'C516322', 'C524883', 'D017632', 'C024860', 'C060089', 'C574123', 'D019086', 'D019815', 'C480026', 'C568507', 'C028330', 'C447082', 'C057013', 'C005277', 'C000611464', 'D020319', 'C051070', 'C044613', 'D009466', 'C562242', 'C469552', 'C081662', 'D011135', 'C000609944', 'D019826', 'C052787', 'C103848', 'C000624217', 'C006253', 'C530263', 'C562238', 'C502587', 'D005609', 'C528245', 'C071945', 'C038243', 'D000077725', 'D008238', 'C532683', 'D005503', 'C531023', 'C477900', 'C530925', 'C105593', 'C525485', 'C560402', 'C517827', 'C003715', 'C021310', 'C000610679', 'C077594', 'C525079', 'C042957', 'C403134', 'C107225', 'D016212', 'C100185', 'C553294', 'D003678', 'D020155', 'C407360', 'C000609888', 'D054833', 'C417690', 'C525246', 'C000589288', 'C085131', 'C544970', 'C482574', 'C408604', 'C065784', 'C000627295', 'C048507', 'C078632', 'C102994', 'C117757', 'C033373', 'C062938', 'D005070', 'C487890', 'C018414', 'C010422', 'C439584', 'C111751', 'C570842', 'C000626478', 'C558507', 'C098123', 'C003305', 'C530605', 'C069616', 'C540581', 'C561056', 'C111116', 'C075603', 'C012125', 'C523410', 'C064848', 'C041636', 'C025603', 'C059021', 'D004369', 'C026164', 'C575152', 'C000722512', 'C578967', 'C015476', 'D000946', 'C506291', 'C111472', 'C511621', 'D000477', 'C561238', 'C046455', 'C000600816', 'C007036', 'C079805', 'C090069', 'C471035', 'D004492', 'C491625', 'C049636', 'C568285', 'C007858', 'C036837', 'C000601538', 'C009338', 'C461657', 'D008456', 'C054084', 'C488176', 'C451163', 'C000598755', 'C061450', 'C488736', 'C092913', 'C030956', 'D042442', 'C061091', 'C071160', 'D016861', 'D014269', 'D007208', 'C027562', 'D007220', 'C469668', 'C505014', 'C121854', 'D013491', 'C492911', 'D034961', 'C000713218', 'C059528', 'C470655', 'C022306', 'C491996', 'C581286', 'C119319', 'C429623', 'D019807', 'C046782', 'C543150', 'C496730', 'C004282', 'C026528', 'C062258', 'C529994', 'C024139', 'C546599', 'C017911', 'C072936', 'C527793', 'D020536', 'C539419', 'C082414', 'C561227', 'C087550', 'C029010', 'C107393', 'C000722770', 'D007004', 'C419543', 'C562099', 'C522300', 'C581774', 'C000604515', 'D009414', 'C512862', 'D019440', 'C000628468', 'C478959', 'C017618', 'C000603501', 'C122128', 'C584586', 'D002491', 'C115495', 'C542919', 'C000602001', 'C000630879', 'C529269', 'C013694', 'C013935', 'C559490', 'C000603149', 'C029823', 'C044779', 'C451780', 'D013657', 'D019344', 'D018690', 'C538779', 'C528580', 'C083828', 'D000903', 'C534679', 'C410026', 'C454676', 'C007377', 'C089665', 'C543534', 'C584298', 'C435023', 'C532363', 'C446946', 'C504897', 'C462953', 'D009465', 'D011848', 'D010831', 'C045598', 'C524616', 'C058808', 'D004786', 'C118220', 'C531547', 'C094635', 'C060048', 'C032278', 'C050408', 'C083998', 'C120727', 'D009947', 'C558736', 'C103325', 'C509369', 'C486592', 'C483241', 'C512273', 'C038599', 'C495433', 'C006012', 'C090812', 'C041797', 'D026541', 'C423582', 'C030358', 'C116674', 'C571832', 'C094150', 'C068314', 'C508388', 'C005296', 'C064507', 'D058787', 'C000594493', 'C514222', 'C000592886', 'D061988', 'D006152', 'C505559', 'C095088', 'D012794', 'C099406', 'C016195', 'C064441', 'C529447', 'C120556', 'C569668', 'C000601052', 'C576677', 'D008773', 'C005177', 'C007139', 'C030544', 'C540575', 'D000924', 'C472885', 'C000615642', 'D011465', 'C434584', 'C582338', 'C000629046', 'C525519', 'C568612', 'C110041', 'C570525', 'C574016', 'C055124', 'C513644', 'C520404', 'D013501', 'D019802', 'C046938', 'D018696', 'C015234', 'C034224', 'C576398', 'C027113', 'D000073396', 'D010972', 'C000543', 'D019289', 'C585533', 'C086427', 'C474557', 'D013956', 'C012375', 'D000094062', 'C103481', 'D000928', 'C029536', 'C549914', 'C044181', 'C425776', 'C054935', 'C559288', 'D002393', 'D000067956', 'C569677', 'D008469', 'C104750', 'C561235', 'C000603489', 'C504648', 'C575425', 'C061640', 'C000655364', 'C496145', 'C025525', 'C036446', 'C099177', 'C018348', 'C553580', 'D009540', 'C544129', 'C091517', 'C499693', 'C481648', 'C473783', 'C508169', 'D018685', 'C571869', 'C049154', 'D003473', 'C090066', 'C513635', 'C540317', 'C498045', 'C116673', 'D000072338', 'C106335', 'C014182', 'C417083', 'C009252', 'D006000', 'C043090', 'C050175', 'C051040', 'D019980', 'C000727867', 'C079391', 'C033093', 'C051732', 'C071787', 'C034613', 'C568518', 'C581009', 'C025086', 'C043105', 'C471210', 'C112353', 'C504605', 'C556398', 'C023833', 'C000707859', 'C035132', 'D011095', 'C070066', 'C022270', 'C515164', 'D000097123', 'C080222', 'C532725', 'C119129', 'C000614591', 'C471071', 'C524106', 'C005023', 'C019427', 'C033674', 'C042771', 'C512706', 'C019429', 'C573098', 'C016096', 'D018683', 'C040668', 'C552929', 'C002850', 'C544130', 'C049352', 'D058425', 'D018894', 'C075597', 'C561591', 'C555919', 'C587358', 'C011126', 'C045076', 'C502525', 'C570631', 'C018968', 'C510784', 'C482648', 'C021270', 'C000717808', 'C028995', 'C513370', 'C101577', 'C000618732', 'D052246', 'C550212', 'C000616979', 'C099343', 'D013434', 'C507418', 'C005056', 'C510327', 'C030735', 'C466800', 'C540329', 'C017501', 'C008884', 'C046062', 'C068538', 'C000632884', 'C569212', 'C048569', 'C539252', 'C545823', 'C000722994', 'D009153', 'C045038', 'C070899', 'D008794', 'C053084', 'C101705', 'C071351', 'D006514', 'C116893', 'C000605453', 'C009709', 'C115124', 'C576414', 'C482199', 'C030692', 'C066902', 'C026832', 'C116996', 'C005087', 'C019309', 'C571008', 'C007695', 'C045139', 'C014289', 'C468283', 'D010975', 'C525063', 'D015064', 'C532415', 'C436947', 'D008299', 'C582521', 'C034008', 'C012134', 'C556295', 'C024957', 'C513485', 'C000588918', 'C508488', 'C000604217', 'C111420', 'D013745', 'C569869', 'C092312', 'C055986', 'C522498', 'C551455', 'C518644', 'C505499', 'C484278', 'C575631', 'C120508', 'C487366', 'C423222', 'C040514', 'C513935', 'C518313', 'C101815', 'C000590659', 'C000708357', 'D004976', 'C492891', 'C058699', 'C114163', 'C498148', 'C025080', 'C013488', 'C008418', 'C036836', 'C120596', 'C455860', 'D006997', 'C415563', 'C478854', 'C075655', 'C515440', 'C534918', 'C107690', 'C513234', 'C533633', 'C078592', 'D062907', 'C532967', 'C514211', 'C033686', 'C034410', 'C082097', 'C440499', 'D008996', 'C450880', 'D005678', 'C530990', 'C002211', 'C520892', 'C549350', 'C113861', 'C045338', 'C008033', 'C066096', 'C500175', 'C477863', 'D013935', 'D011143', 'D017323', 'D013441', 'C004934', 'C423062', 'C552819', 'C509597', 'D019266', 'C551941', 'C503119', 'C572723', 'C521335', 'C072043', 'D000890', 'C035514', 'C574914', 'C475503', 'C016070', 'C083510', 'C517170', 'C529910', 'C524107', 'C578491', 'C461261', 'C538763', 'D010952', 'C078706', 'C025474', 'C030438', 'C000717911', 'D000779', 'C560273', 'C100679', 'C042137', 'C050500', 'C419410', 'C526614', 'C487780', 'C559867', 'D016222', 'C112485', 'D004785', 'C099840', 'C112429', 'C505145', 'D016318', 'C052560', 'C039323', 'C000590846', 'C502805', 'C562048', 'C086910', 'C540172', 'C568520', 'C439677', 'D000097789', 'C055830', 'C507711', 'C000600499', 'C509621', 'C056903', 'C065868', 'C427308', 'C100435', 'C109602', 'C437963', 'C032405', 'C001114', 'C100434', 'C512731', 'C438613', 'C015292', 'C000597233', 'C414214', 'C011890', 'C533469', 'C568960', 'D004998', 'C000708871', 'C472071', 'C067525', 'D010936', 'C562249', 'D006538', 'C077055', 'C521391', 'C568460', 'C548497', 'C483845', 'C526714', 'C031492', 'C061285', 'C519736', 'D014580', 'C004925', 'C416382', 'C000597466', 'C515441', 'C005842', 'D005505', 'C545536', 'C519014', 'C025168', 'C000594853', 'C583916', 'C407102', 'C047587', 'C512408', 'D022681', 'C510582', 'C570019', 'C023850', 'C096326', 'C073427', 'C008999', 'C487680', 'D002723', 'D005397', 'C578079', 'C479119', 'C038293', 'C023677', 'C033901', 'C096278', 'C038335', 'C552254', 'C070189', 'C102521', 'C105065', 'C096713', 'C057147', 'C114198', 'C508955', 'C545880', 'C111601', 'C059510', 'C000589596', 'C539574', 'C558729', 'C109079', 'C116017', 'D005231', 'D000068297', 'C012559', 'C482862', 'C560905', 'C455554', 'C001446', 'C121335', 'C043237', 'C489152', 'C014328', 'C503643', 'C046419', 'C452039', 'C419157', 'C568962', 'C001886', 'C094432', 'C528813', 'C503945', 'C487289', 'C000655265', 'C422565', 'C550454', 'C560681', 'C013332', 'D005232', 'C100126', 'C045533', 'C016957', 'C053519', 'D002136', 'C547325', 'C497935', 'C016262', 'C025602', 'C579694', 'C082137', 'C100973', 'C030408', 'D000214', 'C518368', 'C100698', 'C476437', 'C575632', 'C498130', 'D003280', 'C586406', 'C558403', 'C047981', 'C581572', 'C008757', 'C074968', 'C051283', 'D001312', 'C000626913', 'C479979', 'D006002', 'C045335', 'C518404', 'C531013', 'D017965', 'D020030', 'C040810', 'D015525', 'C433826', 'C553724', 'C093180', 'C000604529', 'C548779', 'C063463', 'D000644', 'C539348', 'C054625', 'C047757', 'C586842', 'C051894', 'C044361', 'C426023', 'C412490', 'C469683', 'C463040', 'C042020', 'D000075242', 'C438036', 'C469937', 'D018394', 'C028491', 'C097284', 'C105826', 'C028952', 'C533761', 'C000611953', 'C110190', 'C000604113', 'D009571', 'C036423', 'C470996', 'C508274', 'C577623', 'C000596174', 'C052983', 'D045762', 'C571006', 'C587536', 'C042971', 'D000067397', 'C093337', 'C005108', 'C517469', 'C117947', 'C027281', 'C006014', 'C072553', 'C115612', 'C570540', 'C045561', 'C524509', 'C061995', 'D004281', 'C474576', 'C558735', 'C529295', 'C465960', 'C064758', 'C000589290', 'C465666', 'C007930', 'C091884', 'C092721', 'C115527', 'C412893', 'C514968', 'C092824', 'C112943', 'C077064', 'C558737', 'C432744', 'D006820', 'C092625', 'D012369', 'D013111', 'C000713467', 'C108729', 'C485023', 'C100640', 'C097240', 'C504503', 'C102282', 'C115626', 'C587540', 'C109479', 'D005520', 'C034857', 'C015559', 'C472349', 'C456520', 'C554289', 'C497101', 'C049246', 'C561230', 'C457234', 'C094644', 'C058691', 'C550547', 'C547124', 'D001962', 'C000590225', 'C000720146', 'C013025', 'C065959', 'C015092', 'D002317', 'C561237', 'C000603033', 'C000609801', 'C041105', 'C032279', 'C000629302', 'C007956', 'D001067', 'C076851', 'C093026', 'C468150', 'C551630', 'C046230', 'C113498', 'C411764', 'C087595', 'C098657', 'C000608435', 'C000626960', 'C107843', 'C523934', 'C552952', 'D000276', 'C086509', 'C576094', 'C470632', 'C090663', 'C443758', 'C065551', 'C401930', 'C062152', 'C401206', 'C044387', 'C560790', 'C028268', 'C022794', 'C586464', 'C562241', 'C583787', 'C019123', 'D004002', 'C008842', 'C524257', 'C518576', 'C570401', 'C051895', 'C509625', 'C545493', 'D002191', 'C001082', 'C515547', 'C050950', 'C070021', 'D020245', 'C067851', 'C470413', 'C000090', 'C575740', 'C515928', 'C031150', 'C041405', 'C577235', 'C016481', 'D000077713', 'C010395', 'C073356', 'C007488', 'C000483', 'C111350', 'C051090', 'C581825', 'C531033', 'C562047', 'C033225', 'C030590', 'C038864', 'D018698', 'C482769', 'C477054', 'C544427', 'C065550', 'C119971', 'C056644', 'C550530', 'C083877', 'C555017', 'C034350', 'C043857', 'C085278', 'C541079', 'C552122', 'C556217', 'C512984', 'C097009', 'C572639', 'C009502', 'C541117', 'C539612', 'C442478', 'C527346', 'D005229', 'C052996', 'D010094', 'C450643', 'C502236', 'C027356', 'C000591741', 'C529219', 'C474696', 'C027756', 'C570013', 'D013656', 'C025145', 'D047248', 'C513163', 'C473894', 'C000718262', 'D000969', 'D015121', 'C018539', 'C514353', 'C041579', 'C545531', 'C097869', 'C078927', 'D015647', 'C083640', 'C054992', 'C415840', 'C092883', 'C083832', 'C121228', 'C040349', 'C058722', 'C576701', 'C511295', 'C059990', 'D007850', 'C493484', 'C051514', 'D010662', 'C076221', 'C032704', 'C026563', 'D011619', 'C030737', 'C539256', 'C000616980', 'C480229', 'D018691', 'C000617228', 'D019695', 'C000589248', 'C008869', 'C556313', 'D020887', 'C408047', 'C516449', 'C513307', 'D010626', 'C552250', 'C000621704', 'C009189', 'C498131', 'D013229', 'C059362', 'D022562', 'C547595', 'C573693', 'D011624', 'C532162', 'C099403', 'C063747', 'C475622', 'C473481', 'D017419', 'C054931', 'C110795', 'C430647', 'C050466', 'D002101', 'C540944', 'C561691', 'C037351', 'C030396', 'C009503', 'C507130', 'C522844', 'C035713', 'D016566', 'C539827', 'C507359', 'D009843', 'C468786', 'C552681', 'C057485', 'C522766', 'C471469', 'C495400', 'C056933', 'C474233', 'C001209', 'C077584', 'D006719', 'C083319', 'D010130', 'C501656', 'C086235', 'C092174', 'C475093', 'C083079', 'C083253', 'D015084', 'C587048', 'C500809', 'C478100', 'C032623', 'C581170', 'C073829', 'C570404', 'C504928', 'C011246', 'C551532', 'C485817', 'D002264', 'D008078', 'C507236', 'C029579', 'C098738', 'C562235', 'C097775', 'C042921', 'C472594', 'C006266', 'C083097', 'C013484', 'C121166', 'D037342', 'D002949', 'D001546', 'C517140', 'C087567', 'C030141', 'C017585', 'C074463', 'C439264', 'C090111', 'C056299', 'C569291', 'C081306', 'C000628268', 'C062898', 'D012102', 'C532889', 'C528911', 'C039737', 'C007936', 'C000602275', 'C467766', 'C116515', 'C017648', 'C000628398', 'C034557', 'D051100', 'C541528', 'C019249', 'C471831', 'D005233', 'C541863', 'D000077557', 'C476105', 'C008105', 'C071554', 'C532849', 'C505646', 'D015699', 'C530803', 'D017639', 'C107666', 'C049811', 'C000603150', 'C506265', 'C459092', 'C006648', 'C402725', 'C438868', 'C000152', 'C067113', 'C575483', 'C117944', 'C009587', 'D007987', 'C521334', 'C472829', 'C552258', 'C431665', 'C000707716', 'C586463', 'C556138', 'C493087', 'C439304', 'C540362', 'C495469', 'C587539', 'C503101', 'D013658', 'C504858', 'C582212', 'C480340', 'C015816', 'C557647', 'C505456', 'C512457', 'D003435', 'C026690', 'C058815', 'C000720090', 'D003281', 'C010030', 'C004817', 'D011078', 'C510942', 'C543211', 'C557967', 'C035538', 'C578458', 'C000723271', 'C006884', 'C471109', 'D008300', 'C047117', 'C007702', 'C400981', 'C569672', 'D004365', 'C420885', 'C420120', 'C534124', 'C051904', 'D044966', 'C501286', 'C577635', 'C568953', 'C000609591', 'C562327', 'C022842', 'C555574', 'C475911', 'C500065', 'C087551', 'C497826', 'C111984', 'C000616933', 'C115283', 'C030106', 'C560272', 'C577936', 'C501618', 'C099414', 'C042579', 'D013424', 'C062737', 'C000616873', 'C052102', 'C000625642', 'C524063', 'C007906', 'C556801', 'C047628', 'D055549', 'C075975', 'C031226', 'C560811', 'C017199', 'C000609924', 'C436083', 'C480124', 'C002470', 'C433788', 'C025043', 'C026915', 'C588213', 'C000731137', 'D012545', 'C503634', 'C550711', 'C522458', 'C517885', 'C539997', 'D005395', 'C540909', 'D004396', 'D012824', 'C488382', 'C024311', 'C095905', 'C568510', 'C542008', 'C034504', 'C108283', 'C526691', 'C475772', 'C540360', 'C084527', 'C021756', 'C108242', 'C033037', 'C077580', 'C457233', 'C100273', 'C073339', 'C544762', 'D007809', 'C532829', 'C035235', 'C552956', 'C515679', 'C061610', 'C559145', 'C572462', 'C070074', 'C588087', 'C532626', 'C413985', 'C513118', 'C043888', 'C541427', 'C055514', 'C007267', 'C114843', 'C569762', 'C521363', 'C072819', 'C452425', 'C495575', 'C070441', 'D026603', 'D005679', 'C475928', 'C037760', 'C540395', 'C067114', 'C015262', 'C438505', 'C013362', 'C556094', 'C409063', 'D011801', 'C518860', 'C055238', 'C543145', 'C030415', 'C521501', 'C568957', 'C036455', 'C488175', 'D010867', 'C011800', 'C016583', 'C000618316', 'C090071', 'C116817', 'C517059', 'C557644', 'D003995', 'C586093', 'C401295', 'C031570', 'D012989', 'D005669', 'D006893', 'C496952', 'D015662', 'C005741', 'C056612', 'D001786', 'C078415', 'C054400', 'C082206', 'C121853', 'D010421', 'D015386', 'C115138', 'C069881', 'C094207', 'C000628397', 'C511794', 'C527191', 'D000083462', 'C061388', 'C060581', 'D001802', 'C546626', 'C492397', 'C118951', 'D053590', 'C062798', 'C015844', 'C403944', 'C009885', 'C582246', 'C084273', 'C510252', 'D005579', 'C519816', 'D018722', 'D054872', 'C562254', 'C106408', 'C093528', 'C568862', 'D006575', 'C006399', 'C000025', 'D005620', 'C112036', 'C488583', 'D000961', 'C008348', 'C042557', 'C025644', 'C030809', 'C556631', 'C084956', 'D043844', 'C542465', 'D001979', 'C516309', 'C030614', 'C004691', 'D007051', 'D020881', 'C496810', 'C503589', 'D058786', 'C085989', 'C061300', 'C108482', 'C000629391', 'C410666', 'C472791', 'C579664', 'C070766', 'C492656', 'C000594115', 'C005969', 'C571532', 'C101290', 'C000603962', 'C528128', 'C495109', 'C121755', 'C082089', 'C103477', 'C044609', 'C035909', 'C494244', 'C517623', 'C511199', 'D000072317', 'C539142', 'C501277', 'C000620717', 'D003894', 'C090324', 'D006064', 'C010654', 'C461658', 'C085470', 'C015885', 'C025541', 'D007692', 'C561590', 'C000707874', 'C504653', 'C488185', 'C496397', 'C528797', 'D001632', 'C080828', 'C546519', 'C000593291', 'C471106', 'C000612485', 'C568512', 'D017378', 'D000156', 'C552889', 'C028431', 'C465253', 'C064146', 'D000956', 'C016868', 'C110417', 'D003311', 'C025591', 'C034318', 'C061482', 'C061621', 'C017475', 'C526278', 'C090535', 'C056213', 'C477639', 'D008692', 'C108481', 'C010410', 'D001147', 'C117455', 'C571448', 'C506598', 'C062764', 'D003879', 'C534983', 'D012765', 'C467893', 'C105294', 'C000624219', 'C103712', 'C034638', 'C559902', 'C509690', 'C502777', 'C000611322', 'C000588809', 'C000602461', 'C018959', 'C068836', 'C069677', 'C000601106', 'C000624958', 'D013723', 'C545501', 'C071318', 'C462999', 'C556685', 'C107087', 'D057886', 'C035879', 'C100040', 'C577249', 'C473484', 'C000599093', 'D013755', 'C000608765', 'C000598830', 'C476008', 'D002351', 'C110414', 'C499148', 'C000721091', 'C480201', 'C581290', 'C547185', 'C045863', 'D000947', 'C086109', 'C416545', 'C555650', 'C521703', 'C083030', 'C058121', 'C561637', 'C000599671', 'C492871', 'D015243', 'C561236', 'C049841', 'C021159', 'D000068257', 'C094299', 'C555859', 'C035638', 'C000588915', 'C498146', 'C505980', 'C009991', 'C543146', 'C545555', 'C064742', 'C411868', 'C556284', 'C047115', 'D006063', 'C109691', 'C002001', 'C453963', 'C588014', 'C530602', 'C481822', 'C561127', 'C037165', 'C040240', 'C025971', 'D019308', 'C052565', 'D004041', 'C083427', 'C011987', 'C039822', 'C571715', 'C410523', 'C438269', 'C062323', 'C432872', 'C099830', 'C000601074', 'C568514', 'C090859', 'C428143', 'D003998', 'D018681', 'C004368', 'D005224', 'C415566', 'C446999', 'D022622', 'C114614', 'D012277', 'C576667', 'D019904', 'C071316', 'C437127', 'C013320', 'C484285', 'C000611223', 'D014302', 'C521389', 'C048336', 'C495194', 'C520611', 'C067372', 'D000179', 'C570533', 'C000298', 'D015095', 'C085632', 'C060076', 'C096429', 'C513113', 'C503118', 'C025351', 'C489863', 'C061750', 'D058647', 'C535237', 'C571005', 'C573672', 'C559687', 'C022894', 'C000595958', 'C095170', 'C520836', 'D000068296', 'C071738', 'C009549', 'D000073893', 'C583883', 'C547593', 'C514762', 'C004961', 'C054895', 'C012301', 'C474722', 'C113106', 'D018074', 'C569633', 'C041260', 'C513003', 'C495482', 'C055331', 'C436283', 'D003346', 'C575637', 'C045596', 'C472987', 'C427703', 'D010795', 'C545735', 'C044611', 'C000618335', 'C530952', 'D000995', 'C021279', 'C470760', 'C110139', 'C423012', 'C081510', 'C476323', 'C114535', 'C487401', 'D001224', 'C021298', 'C066169', 'C432169', 'D000242', 'C483044', 'C071723', 'C513552', 'C516706', 'C539263', 'C058966', 'C559662', 'C069642', 'C046357', 'D004849', 'C000599186', 'C436740', 'C462499', 'D008112', 'C531092', 'C582286', 'C500851', 'C585201', 'C519903', 'C451725', 'D007483', 'C568965', 'C028419', 'D005344', 'C039984', 'D013425', 'C560620', 'C573301', 'C055401', 'C054363', 'C482768', 'C009411', 'C477898', 'C040653', 'C035055', 'C027032', 'C521634', 'D000625', 'C479140', 'C010944', 'C479234', 'C002587', 'C551033', 'C027226', 'C121616', 'C098340', 'C000720141', 'C030266', 'C046508', 'C095427', 'C061126', 'C058544', 'C000590843', 'C058508', 'C496951', 'C506362', 'C066201', 'C535075', 'C429685', 'C506585', 'C055453', 'C087205', 'C001262', 'C476499', 'D006513', 'C104918', 'C056614', 'C030722', 'D010664', 'C475771', 'C060637', 'C059164', 'C415326', 'C572014', 'C484885', 'C485966', 'C541028', 'C505569', 'D050607', 'C095651', 'C549500', 'C095781', 'C548139', 'C508214', 'C555021', 'C568952', 'C447639', 'C014632', 'C000588582', 'C548651', 'C526712', 'C035253', 'C045022', 'C023834', 'C568515', 'C066192', 'C515594', 'C424804', 'C034814', 'C076093', 'C079302', 'C554447', 'C000621705', 'C090785', 'C018231', 'C107241', 'C070058', 'C584368', 'C470754', 'C003498', 'C451261', 'C543849', 'C406082', 'C004768', 'D000077728', 'C000596114', 'C553817', 'C068561', 'C093303', 'C000050', 'C490257', 'C570403', 'C503211', 'C461207', 'C013809', 'C000606936', 'C016133', 'C493322', 'C547344', 'C515196', 'C014184', 'D037742', 'C000619812', 'C524644', 'C492600', 'C035983', 'C487488', 'C000624697', 'C416534', 'C568740', 'C000520', 'C121825', 'C051327', 'C527796', 'C561636', 'C479831', 'C534602', 'C063318', 'C006261', 'C409443', 'C031606', 'C020733', 'C016635', 'D002995', 'C438607', 'C052659', 'C532995', 'C004692', 'C000622346', 'C553926', 'C092887', 'C519728', 'C092108', 'C497115', 'C503120', 'C112100', 'C000603101', 'C008493', 'C009785', 'C018420', 'C441128', 'C110349', 'C559679', 'C496197', 'C583008', 'D015415', 'C000625836', 'C525921', 'C045597', 'C000606994', 'C485699', 'C477718', 'C441602', 'D002083', 'C074606', 'C486784', 'C524606', 'C000610380', 'C501617', 'C118008', 'C032881', 'C033154', 'C054290', 'C013147', 'C051515', 'C407002', 'C009907', 'C045405', 'C000595501', 'D003278', 'C484018', 'C519015', 'C558395', 'D022641', 'C034534', 'C000623391', 'C551454', 'C477642', 'C036567', 'C113118', 'C048214', 'C554428', 'C538985', 'C528863', 'D001498', 'C576038', 'C074277', 'C001036', 'C099060', 'C471843', 'C082229', 'C570545', 'D060766', 'D008698', 'C489251', 'C575635', 'C555802', 'C021974', 'C051450', 'C570834', 'C027981', 'C116108', 'C117466', 'C014118', 'C083095', 'C556297', 'C035549', 'C523946', 'C517629', 'C038967', 'C473710', 'D008744', 'C578354', 'D018712', 'C411320', 'C000602093', 'C587274', 'C002202', 'C517687', 'C010643', 'C029449', 'C524038', 'D004997', 'D005303', 'C015209', 'D064800', 'C095804', 'C003614', 'C043283', 'C022118', 'C479838', 'C530394', 'C498946', 'C074950', 'C058391', 'C102563', 'C415243', 'C021303', 'C046825', 'C025540', 'D006841', 'C403754', 'C482900', 'C027493', 'C046652', 'C018873', 'D018836', 'C581356', 'C433938', 'C011560', 'C120038', 'C474453', 'C534887', 'C020251', 'C045674', 'C479171', 'C016658', 'C403065', 'C413549', 'C104855', 'C000628644', 'C112536', 'C569675', 'C010452', 'C585467', 'C572384', 'C568958', 'C584509', 'C412914', 'C504317', 'C542006', 'C076190', 'C103081', 'C000597234', 'C092028', 'C009548', 'C000627437', 'D013011', 'C093363', 'C107016', 'C017429', 'C575594', 'C534047', 'C550371', 'C499810', 'C527832', 'D012990', 'C576403', 'C031639', 'C418262', 'C082398', 'C058848', 'C083763', 'C530171', 'D005406', 'C106499', 'D005840', 'D000068342', 'C445081', 'C476308', 'C487288', 'D022362', 'C000596149', 'D007608', 'C038639', 'C531961', 'C087867', 'D007539', 'C000596092', 'C000625361', 'C019126', 'C529493', 'C015372', 'C557338', 'D000324', 'D000077322', 'C483161', 'C042310', 'C411297', 'C106876', 'C087651', 'C441201', 'C493177', 'C080866', 'C024626', 'C528421', 'C570743', 'C054917', 'C008391', 'C062246', 'C004454', 'C543477', 'C002585', 'C495230', 'D020723', 'C103958', 'D009539', 'C096680', 'D045785', 'C542674', 'C000598659', 'C000625385', 'C038672', 'C016173', 'C091365', 'C528828', 'C491689', 'C068663', 'C550613', 'C070187', 'C071226', 'D006713', 'C044101', 'C016211', 'C496199', 'C499823', 'C000716407', 'C108531', 'C477941', 'C082823', 'C543636', 'C040890', 'D000266', 'C577942', 'C113042', 'C442781', 'C000588556', 'C031385', 'C000600161', 'C500583', 'C114767', 'C576532', 'C479635', 'C487773', 'C434604', 'C000595640', 'C554440', 'C529696', 'C532858', 'C095476', 'C020891', 'C000627634', 'C558526', 'D009706', 'D013756', 'C064396', 'C093371', 'C000717949', 'C005703', 'C505449', 'D005439', 'C037652', 'C410028', 'C417520', 'D012982', 'C530770', 'C036411', 'C412791', 'C005756', 'C000604114', 'C519132', 'C000625285', 'C014844', 'C560625', 'C008070', 'C072046', 'C559056', 'C558527', 'C013026', 'C549241', 'C472707', 'C487701', 'C093323', 'C525147', 'C477157', 'D026082', 'C097270', 'D000664', 'C085901', 'D007485', 'C025984', 'C101816', 'D002255', 'C046675', 'C090529', 'D003503', 'C558289', 'C005915', 'C000595064', 'C000600817', 'D000094982', 'C525468', 'C534987', 'C438724', 'C523062', 'C046783', 'C412605', 'C049599', 'C507134', 'C548172', 'C562237', 'C534628', 'C080703', 'C420024', 'C040643', 'C090942', 'C545581', 'C032732', 'C035736', 'C444919', 'C000603500', 'D000082082', 'C511643', 'C060436', 'C069833', 'C539431', 'C524754', 'C000599870', 'C083357', 'C530259', 'C510228', 'D019796', 'C092337', 'C560580', 'C000591541', 'C492368', 'C477838', 'C015047', 'C028815', 'C087758', 'C000657350', 'C027563', 'C001945', 'D009829', 'C096620', 'C546861', 'C551027', 'C403256', 'C540445', 'C550755', 'C092152', 'C010161', 'C568959', 'C027285', 'C570534', 'C005939', 'C488633', 'C052614', 'C120943', 'C574385', 'D019812', 'C096898', 'C000611307', 'C049740', 'C000729973', 'C112418', 'C114209', 'D004042', 'C435444', 'C097799', 'C000592522', 'C025170', 'C052091', 'C554291', 'C000625641', 'D006851', 'C000627609', 'C024348', 'C000730074', 'D018686', 'C504377', 'D004966', 'D030421', 'C111716', 'C016797', 'C077420', 'C055802', 'C541384', 'C436944', 'C508569', 'C061950', 'C461648']
cids = {}
for i in tqdm(range(3361,len(ids))):
    val = get_smiles_from_mesh_id(ids[i])
    if(val):
        cids[ids[i]] = val

100%|██████████| 1146/1146 [33:12<00:00,  1.74s/it]


In [12]:
print(cids)
with open('vocab/smiles2.csv', 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    for chem in cids.keys():
        writer.writerow([chem,cids[chem]])

{'D006719': ['251891342'], 'C504928': ['318165561'], 'C551532': ['341092916'], 'C472594': ['255511821'], 'C083097': ['488951980'], 'D037342': ['483924253'], 'D002949': ['468975049'], 'D001546': ['313075816', '368283416', '381978218', '406551047', '438600687', '444168857', '472500269', '491629474', '503359284', '505959656', '507686625', '144221148', '383470503', '441077475', '469495389', '482588368', '486312413', '486327407', '496214073', '500803082', '162226205', '341139152', '386248125', '472964811', '505213563', '126691065', '135208061', '164806696', '363901613', '385781258', '489636919', '11528456', '160693123', '176330831', '404770584', '447275669', '463757036', '472186361', '485437848', '508815024', '508815025', '508815026', '513618945'], 'C090111': ['250136085', '404570999', '404571015'], 'C116515': ['313078253'], 'C000628398': ['377766183'], 'C008105': ['348580463'], 'C506265': ['481106702'], 'C000707716': ['504700665'], 'C586463': ['329585604'], 'C504858': ['254561562'], 'C4009

In [ ]:
for chem in chemicals:
    G.add_node(chem, node_type='chemical')
for gene in genes:
    G.add_node(gene, node_type='gene')
for disease in diseases:
    G.add_node(disease, node_type='disease')

for _, row in df_cg.iterrows():
    G.add_edge(row['ChemicalID'], row['GeneID'], edge_type='chem_gene')
for _, row in df_cd.iterrows():
    G.add_edge(row['ChemicalID'], row['DiseaseID'], edge_type='chem_disease')
for _, row in df_gd.iterrows():
    G.add_edge(row['GeneID'], row['DiseaseID'], edge_type='gene_disease')

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

NameError: name 'chemicals' is not defined

In [5]:
import pandas as pd

# Example DataFrame
df_cd = pd.read_csv('edges/chemdis.csv')
df_gd = pd.read_csv('edges/genedis.csv')
df_cg = pd.read_csv('edges/chemgene.csv')
df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneID'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseID'])  
df_gd = df_gd.drop_duplicates(subset=['GeneID', 'DiseaseID'])
# Get the most frequent element and its count


In [ ]:
top_n = 30
top_elements = df_cd['ChemicalID'].value_counts().head(top_n)

print("Top", top_n, "elements and their counts:")
print(top_elements)

Top 10 elements and their counts:
ChemicalID
C006780    5832
D001564    5567
D013749    5485
D014635    5387
C017947    5235
D000082    5034
D004958    4793
C543008    4700
D016604    4670
C009495    4632
Name: count, dtype: int64


In [ ]:
top_n = 30
top_elements = df_cd['DiseaseID'].value_counts().head(top_n)

print("Top", top_n, "elements and their counts:")
print(top_elements)


Top 10 elements and their counts:
DiseaseID
MESH:D001943    10461
MESH:D011471    10300
MESH:D006528     9734
MESH:D056486     9508
MESH:D006973     9458
MESH:D008106     9255
MESH:D008175     8928
MESH:D009765     8709
MESH:D015179     8537
MESH:D003921     8464
Name: count, dtype: int64


In [ ]:
top_n = 30
top_elements = df_gd['GeneID'].value_counts().head(top_n)

print("Top", top_n, "elements and their counts:")
print(top_elements)


NameError: name 'df_gd' is not defined

In [4]:
import pandas as pd
chems = pd.read_csv('chemids.csv')

chems['type'].value_counts()

type
c    13943
s     3814
Name: count, dtype: int64

In [16]:
chemicals = list(set(df_cg['ChemicalID'].unique()) | set(df_cd['ChemicalID'].unique()))
diseases = list(set(df_gd['DiseaseID'].unique()) | set(df_cd['DiseaseID'].unique()))
genes = list(set(df_cg['GeneSymbol'].unique()) | set(df_gd['GeneSymbol'].unique()))
print(len(chemicals))
print(len(genes))
print(len(diseases))

17795
28680
7296


In [39]:
df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneID'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseName'])  

only_chem = chems[(chems['type'] == 'c')]
print(only_chem.head())
chemicals_m = ['MESH:'+c for c in chemicals]
print(chemicals_m)
subs = list(set(chemicals_m) - set(only_chem['chemID'].unique()))
len(subs)

            chemID       cids type
1     MESH:C070055      35823    c
2     MESH:C024713      12389    c
3  MESH:D000077612     166548    c
4     MESH:C538809  131634859    c
5  MESH:C000626479       7565    c
['MESH:D010665', 'MESH:C049505', 'MESH:D006415', 'MESH:D009012', 'MESH:C057924', 'MESH:C490483', 'MESH:C100796', 'MESH:C008876', 'MESH:C576694', 'MESH:D003503', 'MESH:C032808', 'MESH:D000551', 'MESH:C571806', 'MESH:D013779', 'MESH:C031263', 'MESH:C022246', 'MESH:D047310', 'MESH:C014168', 'MESH:D007550', 'MESH:D017975', 'MESH:C105260', 'MESH:D019980', 'MESH:C455554', 'MESH:C109682', 'MESH:C117776', 'MESH:C509292', 'MESH:C439296', 'MESH:C040442', 'MESH:D008013', 'MESH:C027971', 'MESH:C055368', 'MESH:C000595304', 'MESH:C034857', 'MESH:C478738', 'MESH:C488464', 'MESH:C034744', 'MESH:D002440', 'MESH:C502971', 'MESH:D014238', 'MESH:C000713447', 'MESH:C415327', 'MESH:C000621705', 'MESH:C585278', 'MESH:C024552', 'MESH:C420162', 'MESH:C543889', 'MESH:C004845', 'MESH:C516322', 'MESH:C00062

3852